# Phase 1 — Depth & Surface Normal Estimation (VS Code version)

This notebook is adapted from a Google Colab notebook so it runs locally in VS Code.

**Before running:**
1. Install Python 3.9+ and the VS Code extensions **Python** and **Jupyter**.
2. Open a terminal in this folder and run:
   ```
   pip install -r requirements.txt
   ```
3. Open this file in VS Code, click the kernel picker (top right) and select your Python environment.
4. Run the cells top to bottom (Shift+Enter).

Changes made vs. the Colab version:
- Removed `google.colab` imports (not available outside Colab).
- Replaced `files.upload()` with a local file path (`IMAGE_PATH`) — put your photo in the same folder and set the filename.
- Replaced the Colab JavaScript webcam bridge with OpenCV's native webcam (`cv2.VideoCapture`) and a normal `cv2.imshow` window. Press **q** in the video window to stop the live loop.


In [1]:
import cv2
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
from transformers import pipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware initialized. Running on: {device}")


C:\Users\USER\vision_hackathon\nrw-geometry-engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hardware initialized. Running on: cuda


In [2]:
class ScharrNormalEstimator:
    def __init__(self, height, width, fx=500.0, fy=500.0, cx=None, cy=None, device=device):
        self.H = height
        self.W = width
        self.device = device

        cx = cx if cx is not None else width / 2.0
        cy = cy if cy is not None else height / 2.0

        # 1. Precompute Ray Grid for unprojection
        v, u = torch.meshgrid(
            torch.arange(self.H, device=self.device, dtype=torch.float32),
            torch.arange(self.W, device=self.device, dtype=torch.float32),
            indexing='ij'
        )
        x_rays = (u - cx) / fx
        y_rays = (v - cy) / fy
        z_rays = torch.ones_like(x_rays)
        self.ray_grid = torch.stack([x_rays, y_rays, z_rays], dim=-1)

        # 2. Define Scharr Kernels for Depthwise Convolution
        scharr_x = torch.tensor([[[-3., 0., 3.],
                                  [-10., 0., 10.],
                                  [-3., 0., 3.]]], device=device).unsqueeze(1) / 32.0

        scharr_y = torch.tensor([[[-3., -10., -3.],
                                  [0.,  0.,  0.],
                                  [3.,  10.,  3.]]], device=device).unsqueeze(1) / 32.0

        self.kx = scharr_x.repeat(3, 1, 1, 1)
        self.ky = scharr_y.repeat(3, 1, 1, 1)

    def compute_normals_and_coords(self, depth_tensor):
        depth = depth_tensor.to(self.device)

        # Unproject to 3D points
        points = self.ray_grid * depth.unsqueeze(-1)

        # Reshape for convolution: (Batch=1, Channels=3, H, W)
        P = points.permute(2, 0, 1).unsqueeze(0)

        # Compute Tangents via Scharr filtering
        Tx = F.conv2d(P, self.kx, padding=1, groups=3).squeeze(0).permute(1, 2, 0)
        Ty = F.conv2d(P, self.ky, padding=1, groups=3).squeeze(0).permute(1, 2, 0)

        # Cross Product & Normalization
        normals = torch.cross(Tx, Ty, dim=-1)
        normals = F.normalize(normals, p=2, dim=-1)

        # Pad borders to face camera
        border_vec = torch.tensor([0., 0., 1.], device=self.device)
        normals[0, :] = border_vec
        normals[-1, :] = border_vec
        normals[:, 0] = border_vec
        normals[:, -1] = border_vec

        # Map [-1, 1] to RGB [0, 1]
        rgb_normals = (normals + 1.0) / 2.0
        return rgb_normals.cpu().numpy(), points.cpu().numpy()


## Load a photo from disk

Put an image file (e.g. `street.jpg`) in the same folder as this notebook, then set `IMAGE_PATH` below.


In [3]:
print("Loading Depth Anything V2 Small AI Model...")
depth_estimator = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=0 if torch.cuda.is_available() else -1
)

# Set this to the path of your real-world street photo
IMAGE_PATH = "street.jpg"

import os
uploaded = os.path.exists(IMAGE_PATH)

if uploaded:
    with open(IMAGE_PATH, "rb") as f:
        file_bytes = f.read()
    color_img = Image.open(io.BytesIO(file_bytes)).convert('RGB')

    print("\nAI is calculating geometry...")
    result = depth_estimator(color_img)
    predicted_depth_image = result["depth"]

    # The AI outputs relative depth (bright = close).
    # Invert and scale to simulate a 10-meter physical boundary for math stability.
    depth_raw = np.array(predicted_depth_image, dtype=np.float32)
    depth_raw = (255.0 - depth_raw) / 255.0 * 10.0

    # Apply Edge-Aware Bilateral Filter to smooth micro-noise
    depth_filtered = cv2.bilateralFilter(depth_raw, d=5, sigmaColor=0.1, sigmaSpace=5)

    H, W = depth_filtered.shape
    depth_tensor = torch.from_numpy(depth_filtered)

    print("Calculating surface normals...")
    estimator = ScharrNormalEstimator(H, W, fx=500.0, fy=500.0, device=device)
    normal_rgb, coords = estimator.compute_normals_and_coords(depth_tensor)
    print("Processing Complete!")
else:
    print(f"No image found at '{IMAGE_PATH}'. Put a photo there and set IMAGE_PATH, then re-run this cell.")


Loading Depth Anything V2 Small AI Model...


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 4154.22it/s]

No image found at 'street.jpg'. Put a photo there and set IMAGE_PATH, then re-run this cell.


In [4]:
if uploaded:
    # 1. Render Visual Benchmarks
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].imshow(color_img)
    axes[0].set_title("Original RGB Photo")
    axes[0].axis('off')

    axes[1].imshow(depth_filtered, cmap='magma')
    axes[1].set_title("V2 True Depth Map (Filtered)")
    axes[1].axis('off')

    axes[2].imshow(normal_rgb)
    axes[2].set_title("3D Surface Normals")
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

    # 2. Sample 100 Pixels & Overlay Data
    np.random.seed(42)
    num_samples = 100
    sample_rows = np.random.randint(15, H - 15, size=num_samples)
    sample_cols = np.random.randint(15, W - 15, size=num_samples)

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(normal_rgb)
    ax.set_title(f"{num_samples} Sampled Pixels with Calculated Z (Depth)", fontsize=14, fontweight='bold')
    ax.axis('off')

    print("=" * 65)
    print(f"{'Point':<8} | {'Pixel (Col, Row)':<18} | {'Calculated Z (Depth)':<20}")
    print("=" * 65)

    for i in range(num_samples):
        r, c = sample_rows[i], sample_cols[i]
        x_val, y_val, z_val = coords[r, c]

        ax.plot(c, r, 'ro', markersize=6, markeredgecolor='white', markeredgewidth=1.2)
        ax.text(c + 8, r + 4, f"P{i+1}: Z={z_val:.2f}m", color='yellow', fontsize=8,
                weight='bold', bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

        print(f"P{i+1:<7} | ({c:>4}, {r:>4})          | {z_val:.4f} m")
    print("=" * 65)
    plt.show()


## Live webcam version (local, no Colab JS bridge needed)

This opens your webcam in a native OpenCV window showing the live RGB feed next to the
live 3D surface normals, with the estimated depth (Z) at the center point.

Press **q** with the video window focused to stop.


In [5]:
webcam_estimator = None

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Could not open webcam. Check that a camera is connected and not in use by another app.")
else:
    try:
        with torch.inference_mode():
            while True:
                ret, frame_bgr = cap.read()
                if not ret:
                    continue

                H, W, _ = frame_bgr.shape
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

                # AI Depth Inference
                pil_img = Image.fromarray(frame_rgb)
                result = depth_estimator(pil_img)
                depth_raw = np.array(result["depth"], dtype=np.float32)

                # Normalize and filter
                depth_raw = (255.0 - depth_raw) / 255.0 * 10.0
                depth_filtered = cv2.bilateralFilter(depth_raw, d=5, sigmaColor=0.1, sigmaSpace=5)
                depth_tensor = torch.from_numpy(depth_filtered)

                # Dynamic grid check for resolution changes
                if webcam_estimator is None or webcam_estimator.H != H or webcam_estimator.W != W:
                    webcam_estimator = ScharrNormalEstimator(H, W, fx=300.0, fy=300.0, device=device)

                # Compute Normals & 3D Coordinates
                normal_rgb, coords = webcam_estimator.compute_normals_and_coords(depth_tensor)

                # Extract Z value at center pixel
                center_r, center_c = H // 2, W // 2
                _, _, z_val = coords[center_r, center_c]

                # Convert surface normals from RGB [0,1] to BGR [0,255] for OpenCV drawing
                normal_bgr = cv2.cvtColor((normal_rgb * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

                # Create side-by-side feed (Live RGB | 3D Surface Normals)
                combined_view = np.hstack([frame_bgr, normal_bgr])

                # Draw target point and depth badge on both feeds
                for offset_x in [0, W]:
                    cx = center_c + offset_x
                    cy = center_r

                    cv2.circle(combined_view, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.circle(combined_view, (cx, cy), 10, (0, 255, 255), 2)

                    label = f"Z: {z_val:.2f}m"
                    cv2.putText(combined_view, label, (cx + 14, cy + 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 3, cv2.LINE_AA)
                    cv2.putText(combined_view, label, (cx + 14, cy + 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)

                cv2.imshow("Live RGB | 3D Surface Normals (press q to quit)", combined_view)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Webcam stream closed.")


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Webcam stream closed.
